# Burn severity workflow
version 8

this notebook takes a file containing one or more burn extent polygons and perfoms the burn severity workflow on them. the out put is either single files per polygon or if desiered one merged geojson for all input polygons

note: combining the results of more that roughly 50 fires into one geojson results in a very large file that is more or less unuseable with a desktop computer and this is not adviseable. 

In [14]:
import datacube
import geopandas as gpd
from datacube.utils.cog import write_cog
import os
from datetime import date, datetime, timedelta


from burn_severity_notebook_helper import *

In [15]:
dc = datacube.Datacube(app="Burnt_severity")

## Before we run the code set up out-put locations and file formats

define the folder location where the output polygons will be saved

for locations on your scratch space you don't need to include '/home/jovyan/' in the address

In [16]:
output_folder = 'results'
#change this path if you are saving in a different location.

file_format = 'geojson'
#be default the files are saved as geojsons

merge_into_one_file = True
#change this to True if you want to combine all outputs into one file

## Define input file

this is the file containing the burn extent polygons you wish to perform burn severity mapping on. Ic can be located in the 'gdata1' share drive or on your own scratch space. by default it is pointing at the 'early cut' of polygons provided by Aurora. 

Any (vector) file format will work, Esri Shapefile, geojson, json, geopackage

(if you are using an esri shapefile they come in four seperate file parts; .cpg, .dbf, .prj, .shp. all parts need to sit in the same folder but you point the script at the '.shp' part )


In [17]:
db_table = 'nli_lastboundaries_trigger'

db_columns = [
    "fire_id",
    "fire_name",
    "fire_type",
    "ignition_date",
    "capt_date",
    "capt_method",
    "area_ha",
    "perim_km",
    "state",
    "agency",
    "date_retrieved",
    "date_processed"
]

app_name = 'Burn_Severity'


all_boundaries_in_database = load_polygons_from_database(db_table, db_columns)

Querying polygons from table 'public.nli_lastboundaries_trigger'...
Retrieved 3805 rows from database.


In [18]:
# input_file = '/home/jovyan/gdata1/projects/Hazards/burn_severity/fire_extents_2025-26_summer/all_fires_at_29JAN26.gpkg'
# #define input file

# poly = gpd.read_file(input_file)
# #open as a geopandas geodataframe and have a look at the first few rows

# #let's peek at the first one
# poly.iloc[0]

## check attributes

code asumes your input polygons have the attributes defined by the trigger product data dictionary. Shapefiles do this fun thing where they shorten long attribute names, so if you are using a shapefile we will fix that up now. 
        'fire_id',
        'fire_name',
        'fire_type',
        'ignition_date',
        'capt_date',
        'capt_method',
        'area_ha',
        'perim_km',
        'state',
        'agency'
        'date_retrieved'
        'date_processed'

if your polygon dosn't have the correct attibutes (they can be empty they just need to be there) the script will get Angry at you and not run

        

In [19]:
#test attribute format

poly = test_polygon_attributes(all_boundaries_in_database)

attibutes are long, no need to change


In [20]:
poly

,fire_id,fire_name,fire_type,ignition_date,capt_date,capt_method,area_ha,perim_km,state,agency,date_retrieved,date_processed,geometry
0,102636004,Tallandoon - Yabba Road,Current Burnt Area,2025-11-01,2025-11-03,Satellite Imagery Sentinel,7.8400000000,2.3300000000,VIC,"Vic Department of Energy, Environment and Clim...",2025-11-04,2025-11-11,"MULTIPOLYGON (((147.25051 -36.42283, 147.25047..."
1,629044,"CRESCENT HEAD RD, CRESCENT HEAD",Bushfire,NaT,2025-11-04,None,2.6900000000,0.7900000000,NSW,Rural Fire Service,2025-11-04,2025-11-11,"MULTIPOLYGON (((152.93944 -31.16917, 152.93941..."
2,2b163b9b-2997-4a2e-bb90-30638dbe3ab3,None,None,2025-11-03,2025-11-03,FIREMAPPER,20370.9100000000,137.5000000000,QLD,Qld Fire and Emergency Services,2025-11-04,2025-11-11,"POLYGON ((150.55255 -27.88581, 150.55268 -27.8..."
3,e6e34de9-6158-4ee7-9024-eae94b80c422,None,None,2025-10-29,2025-11-02,FIREMAPPER,1442.6000000000,20.3600000000,QLD,Qld Fire and Emergency Services,2025-11-03,2025-11-11,"POLYGON ((142.79364 -18.38440, 142.79364 -18.3..."
4,243646,Unnamed Fire - 243646,Current Burnt Area,2025-11-02,2025-11-02,Field Intelligence Field Intelligence and Fire...,58.3600000000,3.7500000000,VIC,Vic Country Fire Authority,2025-11-03,2025-11-11,"MULTIPOLYGON (((143.53699 -35.62578, 143.53677..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3800,643764,"BURRA RD, TULLAMORE",Bushfire,NaT,2026-01-26,None,75.1300000000,4.7000000000,NSW,Rural Fire Service,2026-01-26,2026-02-02,"MULTIPOLYGON (((147.63373 -32.51616, 147.63358..."
3801,9D46A732-FFA8-43BF-A1AB-772DB1AB5588,None,None,2026-01-24,2026-01-24,MICA,0.1300000000,0.2600000000,QLD,Qld Fire and Emergency Services,2026-01-25,2026-02-02,"POLYGON ((153.30072 -27.71577, 153.30074 -27.7..."
3802,3ED1783F-ECE0-49AC-A245-9776587680B1,None,None,2026-01-24,2026-01-24,MICA,0.0300000000,0.1100000000,QLD,Qld Fire and Emergency Services,2026-01-25,2026-02-02,"POLYGON ((153.33434 -27.78294, 153.33416 -27.7..."
3803,102645027,Rich River - Mills Road 3,Current Burnt Area,2026-01-13,2026-01-08,AIG Observation,236.4500000000,16.8300000000,VIC,"Vic Department of Energy, Environment and Clim...",2026-01-14,2026-02-02,"MULTIPOLYGON (((148.66392 -37.52191, 148.66337..."


## pre-filtering

pre-filtering steps on polygons:
- remove any polygons with no area or smaller than 1 ha
- check for duplicates for the same fire and perfrom spatial join on them


In [21]:
#filter on size 
poly = poly.drop(poly[poly.area_ha < 1].index)

#list all fire 'id' names
list_of_fires = list(set(poly['fire_id']))

#perform spatial dissolve on overlapping polygons with same id
dissolved_duplicates = perform_spatial_dissolve(poly, list_of_fires)

In [22]:
dissolved_duplicates.iloc[0]

geometry          POLYGON ((148.069602161 -29.993028229, 148.071...
fire_id                                                      631019
fire_name                                    KAMILAROI HWY, WALGETT
fire_type                                                  Bushfire
capt_method                                                    None
area_ha                                                6.9100000000
perim_km                                               1.2100000000
state                                                           NSW
agency                                           Rural Fire Service
date_retrieved                                  2025-11-18 00:00:00
ignition_date                                                   NaN
capt_date                                       2025-11-18 00:00:00
date_processed                                           2025-11-25
Name: 0, dtype: object

## Filter by age

we only want to run our process of files older that 60 days. that is filres that have been extinguished for more than 60 days


In [23]:
today_date = date.today()

cutoff_date = today_date - timedelta(days=65)

mature_fires = dissolved_duplicates[dissolved_duplicates.date_processed <= cutoff_date]

#make all nan 0 to eliminate cate's date filtering headache!!
mature_fires = mature_fires.fillna(0)

In [24]:
mature_fires.iloc[0]

geometry          POLYGON ((148.069602161 -29.993028229, 148.071...
fire_id                                                      631019
fire_name                                    KAMILAROI HWY, WALGETT
fire_type                                                  Bushfire
capt_method                                                       0
area_ha                                                6.9100000000
perim_km                                               1.2100000000
state                                                           NSW
agency                                           Rural Fire Service
date_retrieved                                  2025-11-18 00:00:00
ignition_date                                                     0
capt_date                                       2025-11-18 00:00:00
date_processed                                           2025-11-25
Name: 0, dtype: object

In [25]:
# THIS_fire = gpd.GeoDataFrame(mature_fires.iloc[0].to_frame().T, geometry='geometry', crs=mature_fires.crs)

# Severity_polygons, debug_layer = map_burn_severity(THIS_fire)

In [26]:
# Severity_polygons

## map burn severity


In [27]:

#this essentially creates a numerical list of each dissolved fire
for fires in range(0,len(mature_fires.index)):

    
    #select each polygon one-by -one
    fire_polygon = gpd.GeoDataFrame(mature_fires.iloc[fires].to_frame().T, geometry='geometry', crs=mature_fires.crs)

    #make a sting which is fire id and position in gpd to not over-write re-use of ids in QLD
    fire_id_forsave = f'{fire_polygon.fire_id.iloc[0]}_{fires}'
    
    #run burn severity mapping
    Severity_polygons, debug_layer = map_burn_severity(fire_polygon)

       # #save_debug layer to file
    write_cog(debug_layer.compute(), f'{output_folder}/DEA_burn_severity_debug_{fire_id_forsave}.tif',
             overwrite=True)

    Severity_polygons.to_file(f'{output_folder}/DEA_burn_severity_{fire_id_forsave}.{file_format}')
    
    print( f'I finished processing number {fires}')
    


CONDUCTING BURN SEVERITY MAPPING                                             geometry fire_id  \
0  POLYGON ((148.06960 -29.99303, 148.07115 -29.9...  631019   

                fire_name fire_type capt_method       area_ha      perim_km  \
0  KAMILAROI HWY, WALGETT  Bushfire           0  6.9100000000  1.2100000000   

  state              agency       date_retrieved ignition_date  \
0   NSW  Rural Fire Service  2025-11-18 00:00:00             0   

             capt_date date_processed  
0  2025-11-18 00:00:00     2025-11-25  
date as string is 0
date as string is 2025-11-18 00:00:00
date as string is 2025-11-25
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Counting good quality pixels for each time step using s2cloudless
Filtering to 5 out of 5 time steps with at least 99.0% good quality pixels
Applying s2cloudless pixel quality/cloud mask
Returning 5 time steps as a dask array
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir_1', 'nbart_ni

KeyboardInterrupt: 

## Run optional merge of geoploygons
if selected this will save the merged polygons in the Same Folder as the individual ones but with the name 'DEA_burn_severity_MERGED'

In [ ]:
if merge_into_one_file == True:
    
    # List all GeoJSON files in the directory
    geojson_files = [f for f in os.listdir(output_folder) if f.endswith(file_format)]
    
    
    # Read and concatenate all GeoDataFrames
    gdfs = []
    for file in geojson_files:
        filepath = os.path.join(output_folder, file)
        gdf = gpd.read_file(filepath)
        gdfs.append(gdf)
    
    # Merge into one GeoDataFrame
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    
    # # Save to one big GeoJSON
    merged_gdf.to_file(f"{output_folder}/DEA_burn_severity_MERGED.{file_format}")
 